# 00 — Análise exploratória (RSNA-STR PE 2020) — G2

Roda no **Kaggle Notebook** com a competição adicionada como Input (CPU basta).
Objetivos: contagens de exames/pacientes/cortes, distribuição de rótulos (corte e exame),
metadados DICOM relevantes (fabricante, espessura, espaçamento, rescale, compressão),
verificação **1 exame = 1 paciente** e exemplos visuais do pré-processamento.
Figuras e tabelas são salvas em `results/eda/`.

In [ ]:
# === Setup (Kaggle) ==========================================================
# Pré-requisitos: aceitar as regras da competição "RSNA STR Pulmonary Embolism Detection"
# e adicioná-la como Input deste notebook (+ Add Input -> Competitions).
import os, sys, subprocess, shutil
REPO_URL = "https://github.com/silvarawenne/ap2-topicos.git"
REPO_BRANCH = "main"                      # troque se estiver usando outra branch
REPO = "/kaggle/working/ap2-topicos"
CODE_DATASET = "/kaggle/input/ap2-topicos-code"   # alternativa sem Internet: repo enviado como Dataset

if not os.path.exists(REPO):
    if os.path.exists(CODE_DATASET):
        shutil.copytree(CODE_DATASET, REPO)
    else:  # exige "Internet on" nas configurações do notebook
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-kaggle.txt"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
print("repo:", REPO, "| dados:", os.listdir("/kaggle/input"))

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from joblib import Parallel, delayed
import pydicom
from pe_baseline.config import load_config, data_path
from pe_baseline.data import load_train, exam_table, STUDY, SERIES, SOP
cfg = load_config()
OUT = Path("results/eda"); OUT.mkdir(parents=True, exist_ok=True)
train = load_train(data_path(cfg, "train.csv"))
exams = exam_table(train)
print(f"cortes: {len(train):,} | exames: {exams[STUDY].nunique():,} | séries: {train[SERIES].nunique():,}")
print("séries por exame:", exams["n_series"].value_counts().to_dict())

## Rótulos: nível de corte e de exame

In [ ]:
lab = exams["label3"].value_counts().rename("exames").to_frame()
lab["%"] = (100 * lab["exames"] / lab["exames"].sum()).round(1)
display(lab)
print(f"prevalência de cortes positivos (todos os exames): {train.pe_present_on_image.mean():.4f}")
pos = exams[exams.pe == 1]
print(f"prevalência de cortes positivos dentro de exames positivos: {pos.n_pos_slices.sum() / pos.n_slices.sum():.4f}")
sub = ["central_pe","leftsided_pe","rightsided_pe","chronic_pe","acute_and_chronic_pe","rv_lv_ratio_gte_1","rv_lv_ratio_lt_1"]
qa = ["qa_motion","qa_contrast","flow_artifact","true_filling_defect_not_pe"]
tab = pd.concat([pos[sub].mean(), exams[qa].mean()]).rename("%").to_frame()
tab["base"] = ["exames com TEP"] * len(sub) + ["todos os exames"] * len(qa)
display(tab.assign(**{"%": (100 * tab["%"]).round(1)}))
tab.to_csv(OUT / "label_prevalence.csv"); lab.to_csv(OUT / "exam_labels.csv")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(exams.n_slices, bins=60); ax[0].set_title("cortes por exame"); ax[0].set_xlabel("n cortes")
ax[1].hist(pos.n_pos_slices / pos.n_slices, bins=40, color="tab:red")
ax[1].set_title("fração de cortes positivos (exames com TEP)")
lab["exames"].plot.bar(ax=ax[2], color=["tab:blue","tab:red","tab:gray"]); ax[2].set_title("rótulo de exame")
plt.tight_layout(); plt.savefig(OUT / "fig_eda_distributions.png", dpi=150); plt.show()

## Metadados DICOM (1 cabeçalho por exame, todos os exames)
Verifica: PatientID único por exame (justifica partir por exame = partir por paciente),
fabricante, kVp, espessura, espaçamento de pixel, RescaleSlope/Intercept e compressão (TransferSyntax).

In [ ]:
def header(study, series, sop):
    p = data_path(cfg, "train", study, series, f"{sop}.dcm")
    ds = pydicom.dcmread(str(p), stop_before_pixels=True)
    g = lambda k: getattr(ds, k, None)
    ps = g("PixelSpacing")
    return {STUDY: study, "PatientID": str(g("PatientID")), "Manufacturer": str(g("Manufacturer")),
            "KVP": float(g("KVP")) if g("KVP") is not None else np.nan,
            "SliceThickness": float(g("SliceThickness")) if g("SliceThickness") is not None else np.nan,
            "PixelSpacing": float(ps[0]) if ps else np.nan, "Rows": g("Rows"), "Columns": g("Columns"),
            "RescaleSlope": float(g("RescaleSlope") or 1), "RescaleIntercept": float(g("RescaleIntercept") or 0),
            "TransferSyntax": str(ds.file_meta.TransferSyntaxUID.name)}
first = train.groupby(STUDY).first().reset_index()
meta = pd.DataFrame(Parallel(n_jobs=-1)(delayed(header)(r[STUDY], r[SERIES], r[SOP]) for _, r in first.iterrows()))
meta.to_csv(OUT / "dicom_meta_per_exam.csv", index=False)
print("PatientID únicos:", meta.PatientID.nunique(), "de", len(meta), "exames")
for c in ["Manufacturer", "TransferSyntax", "RescaleIntercept", "RescaleSlope", "Rows"]:
    print(f"\n{c}:\n", meta[c].value_counts().head(8).to_string())
display(meta[["KVP","SliceThickness","PixelSpacing"]].describe().round(2))

## Exemplo visual: janelas e máscaras do pré-processamento

In [ ]:
from pe_baseline.dicom_io import read_slice, window
from pe_baseline.preprocess import resize_hu, compute_masks
pre = cfg["preprocess"]
rng = np.random.default_rng(cfg["seed"])
pos_rows = train[train.pe_present_on_image == 1].sample(2, random_state=cfg["seed"])
neg_rows = train[train[STUDY].isin(exams.loc[exams.label3 == "negative", STUDY])].sample(1, random_state=cfg["seed"])
rows = pd.concat([pos_rows, neg_rows])
fig, axes = plt.subplots(len(rows), 4, figsize=(16, 4 * len(rows)))
for i, (_, r) in enumerate(rows.iterrows()):
    sl = read_slice(data_path(cfg, "train", r[STUDY], r[SERIES], f"{r[SOP]}.dcm"), tuple(pre["hu_clip"]))
    hu = resize_hu(sl.hu, pre["out_size"]); m = compute_masks(hu, pre)
    for j, (name, (L, W)) in enumerate(pre["windows"].items()):
        axes[i, j].imshow(window(hu, L, W), cmap="gray"); axes[i, j].set_title(f"{name} L{L}/W{W}")
    axes[i, 3].imshow(window(hu, *pre["windows"]["ctpa"]), cmap="gray")
    axes[i, 3].contour(m.roi, [0.5], colors="yellow", linewidths=0.7)
    axes[i, 3].contour(m.vessel, [0.5], colors="cyan", linewidths=0.4)
    axes[i, 3].contour(m.defect, [0.5], colors="red", linewidths=0.8)
    axes[i, 3].set_title(f"ROI/vaso/falha | rótulo corte={r.pe_present_on_image}")
    for a in axes[i]: a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "fig_eda_preprocessing.png", dpi=130); plt.show()

## Partição fixa e amostra do TP1

In [ ]:
!python scripts/01_make_split_and_sample.py
display(pd.read_csv("results/sample_counts.csv"))

In [ ]:
# Baixe estes arquivos (aba Output) e versione no GitHub: splits/*.csv e results/eda/*
!ls -la splits results/eda